# Multi-Source Visual Search Engine (MSSearch) Diagnostic Notebook
## Multi-Image Candidate Pool ($N \ge 1$), Distributed Evidence ($K \ge 1$) & Open-Set Multi-Answer Inspection

This diagnostic notebook demonstrates and instruments **MSSearch** on a multi-source visual environment conforming to the [Annotation Charter](file:///d:/Project/mysite/docs/experiments/multisource/Annotation%20Charter.md):
- **Candidate Pool $\mathcal{C} = \{I_1, I_2, \dots, I_N\}$**: The target high-resolution image is randomly placed alongside $N-1$ distractor images sampled within a localized index window ($\pm 5$) from the dataset.
- **Budget Scaling ($N \times \text{budget}$)**: Stage 4 SAM3 patch scan caps, latent speculative queries, and retention limits scale linearly with $N$ (`16 * N`, `8 * N`, `16 * N`) to preserve full search coverage across multiple sources.
- **Open-Set Multi-Answer Generation**: Stage 6 collision-free early consensus exit is deactivated (`enable_early_consensus_exit=False`), reading all candidate crops across all $N$ images. Extracted facts are semantically deduplicated and consolidated into semicolon-separated natural language (e.g. `red; blue; yellow`), matching the closed-world complete-answer contract.


In [ ]:
# ==============================================================================
# CELL 1: ENVIRONMENT SETUP, GITHUB REPO SYNC & HARDWARE TELEMETRY
# ==============================================================================
import os
import sys
import gc
import json
import re
import time
import subprocess
import importlib
import urllib.request
from typing import List, Dict, Tuple, Optional, Any

import numpy as np
import torch
import torch.nn as nn
from PIL import Image, ImageDraw, ImageFont
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches

REPO_DIR = "/kaggle/working/MSSearch"
github_token = None

# Retrieve GitHub Access Token from Kaggle User Secrets
try:
    from kaggle_secrets import UserSecretsClient
    user_secrets = UserSecretsClient()
    github_token = user_secrets.get_secret("MaCVi")
    print("[SETUP] Successfully retrieved GitHub token from Kaggle User Secret 'MaCVi'.")
except Exception as e:
    github_token = os.environ.get("MaCVi") or os.environ.get("GITHUB_TOKEN")

if not os.path.exists(REPO_DIR):
    if github_token:
        print(f"[SETUP] Cloning MSSearch repository using authenticated token into {REPO_DIR}...")
        clone_url = f"https://{github_token}@github.com/tothanhdat2006/MSSearch.git"
    else:
        print(f"[SETUP] 'MaCVi' secret not found. Attempting unauthenticated clone into {REPO_DIR}...")
        clone_url = "https://github.com/tothanhdat2006/MSSearch.git"
    
    ret = subprocess.run(["git", "clone", clone_url, REPO_DIR], capture_output=True, text=True)
    if ret.returncode == 0:
        print(f"[SETUP] Successfully cloned MSSearch repository into {REPO_DIR}")
    else:
        safe_err = ret.stderr.replace(github_token, "******") if github_token else ret.stderr
        print(f"[SETUP] Git clone failed: {safe_err}")
else:
    print(f"[SETUP] MSSearch repository already exists at {REPO_DIR}. Pulling latest changes from main...")
    ret = subprocess.run(["git", "-C", REPO_DIR, "pull", "origin", "main"], capture_output=True, text=True)
    if ret.returncode == 0:
        print(f"[SETUP] Repository successfully updated to latest commit: {ret.stdout.strip()}")

# Ensure REPO_DIR is at the front of sys.path
if REPO_DIR not in sys.path:
    sys.path.insert(0, REPO_DIR)
elif sys.path[0] != REPO_DIR:
    sys.path.remove(REPO_DIR)
    sys.path.insert(0, REPO_DIR)

importlib.invalidate_caches()
# Unload previously cached mssearch modules so git pull takes immediate effect
for mod in list(sys.modules.keys()):
    if mod.startswith("mssearch"):
        del sys.modules[mod]

# Ensure BPE vocab file is present for SAM3 tokenizer
bpe_target = os.path.join(REPO_DIR, "assets", "bpe_simple_vocab_16e6.txt.gz")
if not os.path.exists(bpe_target):
    os.makedirs(os.path.dirname(bpe_target), exist_ok=True)
    bpe_url = "https://raw.githubusercontent.com/openai/CLIP/main/clip/bpe_simple_vocab_16e6.txt.gz"
    print(f"[SETUP] Downloading missing BPE vocab to {bpe_target}...")
    try:
        urllib.request.urlretrieve(bpe_url, bpe_target)
        print("[SETUP] Successfully downloaded BPE vocab asset.")
    except Exception as e:
        print(f"[SETUP WARNING] BPE download failed: {e}")

# Install required dependencies
!pip install -q "transformers>=4.45.0" "accelerate>=0.26.0" "tabulate" iopath decord ftfy scikit-image scikit-learn networkx

# Hardware Telemetry
device_count = torch.cuda.device_count()
print("=" * 75)
print(f"PyTorch Version:  {torch.__version__}")
print(f"CUDA Available:   {torch.cuda.is_available()}")
print(f"GPU Device Count: {device_count}")
for i in range(device_count):
    prop = torch.cuda.get_device_properties(i)
    free_mem, total_mem = torch.cuda.mem_get_info(i)
    print(f"  [GPU {i}] {prop.name} | Total: {total_mem/(1024**3):.2f} GiB | Free: {free_mem/(1024**3):.2f} GiB")
has_sdpa = hasattr(torch.nn.functional, 'scaled_dot_product_attention')
print(f"PyTorch SDPA Available: {has_sdpa}")
print("=" * 75)


In [ ]:
# ==============================================================================
# CELL 2: MULTI-SOURCE DATASET INGESTION & N-IMAGE CANDIDATE POOL SAMPLING
# ==============================================================================
import os
import json
import random
from PIL import Image, ImageDraw, ImageFont
import matplotlib.pyplot as plt

HRBENCH_SEARCH_DIRS = [
    "D:/Research/VisualSearch/hr-bench_8k",
    "d:/Research/VisualSearch/hr-bench_8k",
    "/kaggle/input/datasets/thnhtt/vs-dataset/hr-bench_8k/mnt/sdb/llp/datasets/zoom_eye_data/zoom_eye_data/hr-bench_8k",
    "/kaggle/input/datasets/thnhtt/vs-dataset/hr-bench_8k",
    "/kaggle/input/datasets/thnhtt/vs-dataset/hr-bench_4k",
    "/kaggle/input/hr-bench-8k",
    "/kaggle/input/hr-bench-4k",
    "docs/experiments/multisource/data/hr-bench_8k",
    "docs/experiments/freeform/ICML26-CVSearch/data/hr-bench_8k"
]

HRBENCH_BASE_DIR = None
for d in HRBENCH_SEARCH_DIRS:
    if os.path.exists(d):
        HRBENCH_BASE_DIR = d
        break

if HRBENCH_BASE_DIR is None:
    raise FileNotFoundError(f"[Dataset Error] Could not find HRBench directory in any candidate path: {HRBENCH_SEARCH_DIRS}")

HRBENCH_ANNOTATION_PATH = os.path.join(HRBENCH_BASE_DIR, "annotation_hr-bench_8k.json")
if not os.path.exists(HRBENCH_ANNOTATION_PATH):
    HRBENCH_ANNOTATION_PATH = os.path.join(HRBENCH_BASE_DIR, "annotation_hr-bench_4k.json")

print(f"[Dataset Discovery] Using HRBench root: {HRBENCH_BASE_DIR}")
print(f"[Dataset Discovery] Loading annotations: {HRBENCH_ANNOTATION_PATH}")

with open(HRBENCH_ANNOTATION_PATH, "r", encoding="utf-8") as f:
    hrbench_data = json.load(f)

print(f"[Dataset] Ingested {len(hrbench_data)} total samples.")

# ------------------------------------------------------------------------------
# MULTI-SOURCE CONFIGURATION KNOBS
# ------------------------------------------------------------------------------
SAMPLE_INDEX = 0             # Primary target sample index
NUM_CANDIDATE_IMAGES = 3     # Total N candidate images (1 target + N-1 distractors)
WINDOW_SIZE = 5              # Local index sampling window: [SAMPLE_INDEX - 5, SAMPLE_INDEX + 5]
RANDOM_SEED = 42             # Deterministic sampling & shuffling seed

target_sample = hrbench_data[SAMPLE_INDEX]
real_question = target_sample.get('question', '')
cycle_idx = target_sample.get('cycle_category', 0)
gt_answer = target_sample.get('answer', '')

def resolve_gt_letter(answer_obj, cycle_idx=0):
    if isinstance(answer_obj, list):
        if 0 <= cycle_idx < len(answer_obj):
            return str(answer_obj[cycle_idx]).strip().upper()
        return str(answer_obj[0]).strip().upper() if answer_obj else ""
    return str(answer_obj).strip().upper()

gt_letter = resolve_gt_letter(gt_answer, cycle_idx)
raw_options = target_sample.get('options', {})

print("=" * 75)
print(f"[Target Sample {SAMPLE_INDEX}] Question: '{real_question}'")
print(f"[Target Sample {SAMPLE_INDEX}] Ground Truth: Option '{gt_letter}'")
print("=" * 75)

# Load Target Image
target_img_rel = target_sample.get('input_image', '')
target_img_path = os.path.join(HRBENCH_BASE_DIR, target_img_rel)
if not os.path.exists(target_img_path):
    target_img_path = target_img_rel
target_canvas = Image.open(target_img_path).convert("RGB")

# ------------------------------------------------------------------------------
# Sample (N - 1) Distractor Images from [SAMPLE_INDEX - 5, SAMPLE_INDEX + 5]
# ------------------------------------------------------------------------------
total_samples = len(hrbench_data)
min_win = max(0, SAMPLE_INDEX - WINDOW_SIZE)
max_win = min(total_samples - 1, SAMPLE_INDEX + WINDOW_SIZE)

candidate_distractor_indices = [
    idx for idx in range(min_win, max_win + 1) if idx != SAMPLE_INDEX
]

# If window has fewer than N-1 items, expand gracefully
if len(candidate_distractor_indices) < (NUM_CANDIDATE_IMAGES - 1):
    candidate_distractor_indices = [
        idx for idx in range(total_samples) if idx != SAMPLE_INDEX
    ]

rng = random.Random(RANDOM_SEED)
sampled_distractor_indices = rng.sample(candidate_distractor_indices, NUM_CANDIDATE_IMAGES - 1)
print(f"[Sampling] Distractor indices sampled within +/- {WINDOW_SIZE}: {sampled_distractor_indices}")

# Build initial candidate pool items: (is_target, sample_idx, image_pil, rel_path)
pool_items = [
    {"is_target": True, "sample_idx": SAMPLE_INDEX, "image": target_canvas, "path": target_img_rel}
]

for d_idx in sampled_distractor_indices:
    d_sample = hrbench_data[d_idx]
    d_rel = d_sample.get('input_image', '')
    d_path = os.path.join(HRBENCH_BASE_DIR, d_rel)
    if not os.path.exists(d_path):
        d_path = d_rel
    d_img = Image.open(d_path).convert("RGB")
    pool_items.append({
        "is_target": False, "sample_idx": d_idx, "image": d_img, "path": d_rel
    })

# Randomly shuffle candidate images to simulate unordered candidate pool C
rng.shuffle(pool_items)

candidate_images = [item["image"] for item in pool_items]
target_pool_idx = next(i for i, item in enumerate(pool_items) if item["is_target"])

print(f"[Candidate Pool C] Total N = {len(candidate_images)} images constructed.")
print(f"[Candidate Pool C] Target image is positioned at Pool Index: {target_pool_idx}")
for idx, item in enumerate(pool_items):
    tag = "TARGET" if item["is_target"] else "DISTRACTOR"
    print(f"  - Image {idx}: [{tag}] Sample #{item['sample_idx']} | Path: '{item['path']}' | Size: {item['image'].size}")

# ------------------------------------------------------------------------------
# Side-by-Side Thumbnail Visualization of Candidate Pool C
# ------------------------------------------------------------------------------
fig, axes = plt.subplots(1, len(candidate_images), figsize=(5 * len(candidate_images), 5))
if len(candidate_images) == 1:
    axes = [axes]

for i, (ax, item) in enumerate(zip(axes, pool_items)):
    thumb = item["image"].copy()
    thumb.thumbnail((512, 512))
    ax.imshow(thumb)
    is_t = item["is_target"]
    color = "forestgreen" if is_t else "royalblue"
    label = f"Image {i} [TARGET]\nSample #{item['sample_idx']}" if is_t else f"Image {i} [DISTRACTOR]\nSample #{item['sample_idx']}"
    ax.set_title(label, fontsize=11, fontweight="bold", color=color)
    ax.axis("off")
    for spine in ax.spines.values():
        spine.set_edgecolor(color)
        spine.set_linewidth(3)

plt.suptitle(f"Candidate Image Pool C (N = {len(candidate_images)}) for Question:\n'{real_question}'", fontsize=12, fontweight="bold")
plt.tight_layout()
plt.show()

# Retain backward-compatible reference for single-image inspection
active_canvas = candidate_images[target_pool_idx]


In [ ]:
# ==============================================================================
# CELL 3: FOUNDATION MODEL INITIALIZATION & MULTI-SOURCE CONFIGURATION
# ==============================================================================
import os
import sys
import gc
import torch
from PIL import Image

from mssearch.config import MSSearchConfig
from mssearch.models.modeling_sam3 import sam3_inference
from mssearch.models.modeling_siglip import SigLIPScorer
from mssearch.models.modeling_qwenvl import ModelQwenVL
from mssearch.stages.stage1_parser import parse_question_to_active_objects, ParsedQuestion
from mssearch.stages.stage2_fast_path import run_direct_global_assessment, DirectAnswerResult, run_tool_based_fast_path, FastPathResult, compute_composite_bbox
from mssearch.stages.stage3_partitioning import partition_candidate_images, compute_spatial_entropy, partition_oversized_box, split_4_subpatches, partition_into_sam3_sgap, BoundedPatch
from mssearch.stages.stage4_grouping_gating import run_feature_grouping_and_gating, group_patches_by_cosine_similarity
from mssearch.stages.stage5_screening import run_resolution_screening, CandidateCrop
from mssearch.stages.stage6_foveated_reader import run_foveated_vlm_reading, is_negative_or_unanswerable, deduplicate_answers
from mssearch.MSSearch import MultiSourceSearchEngine, get_mssearch_response

# 1. Discover SAM3 Weights
SAM3_CHECKPOINT_PATH = "/kaggle/input/datasets/thnhtt/vs-sam3/sam3/sam3/sam3.pt"
if not os.path.exists(SAM3_CHECKPOINT_PATH):
    for root, _, files in os.walk("/kaggle/input"):
        for f in files:
            if f == "sam3.pt":
                SAM3_CHECKPOINT_PATH = os.path.join(root, f)
                break
        if os.path.exists(SAM3_CHECKPOINT_PATH):
            break

if not os.path.exists(SAM3_CHECKPOINT_PATH):
    raise FileNotFoundError(f"[Model Discovery Error] SAM3 checkpoint not found at '{SAM3_CHECKPOINT_PATH}'.")

print(f"[Model Discovery] SAM3 Weights: {SAM3_CHECKPOINT_PATH}")

# 2. Asymmetric Dual-GPU Allocation
num_gpus = torch.cuda.device_count()
sam3_device = "cuda:1" if num_gpus > 1 else ("cuda:0" if num_gpus > 0 else "cpu")
siglip_device = "cuda:0" if num_gpus > 0 else "cpu"
qwen_device = "cuda:0" if num_gpus > 0 else "cpu"

print(f"[Hardware Topology] Detected {num_gpus} GPUs. Allocation:")
print(f"  - SAM3 Engine      : {sam3_device}")
print(f"  - SigLIP-SO400M    : {siglip_device}")
print(f"  - Qwen2.5-VL Engine: {qwen_device}")

# 3. Initialize Production Models (Strict Fail-Fast, No Fallbacks, No Aliases)
print(f"[Loading] SAM3 Local Segmenter on {sam3_device}...")
sam3_engine = sam3_inference(model_path=SAM3_CHECKPOINT_PATH, device=sam3_device)
print("  [Loaded] SAM3 Local Segmenter Engine")

print(f"[Loading] SigLIP-SO400M on {siglip_device}...")
siglip_engine = SigLIPScorer(model_name="google/siglip-base-patch16-384", device=siglip_device, torch_dtype=torch.float16)
print("  [Loaded] SigLIP Saliency & Verification Engine")

print(f"[Loading] Qwen2.5-VL-7B-Instruct on {qwen_device}...")
qwen_engine = ModelQwenVL(model_path="Qwen/Qwen2.5-VL-7B-Instruct", device=qwen_device, torch_dtype=torch.float16, patch_scale=1.2)
print("  [Loaded] Qwen2.5-VL Multi-Scale Reader Engine")

NLP_MODEL_PATH = "en_core_web_sm"
try:
    import spacy
    nlp_parser = spacy.load(NLP_MODEL_PATH)
    print("  [Loaded] spaCy NLP Grammar Engine")
except Exception:
    nlp_parser = None

# ------------------------------------------------------------------------------
# 4. Master Multi-Source Configuration (N * Budget & Multi-Answer Mode)
# ------------------------------------------------------------------------------
config_track_a = MSSearchConfig(
    # Multi-Source & Multi-Answer Knobs (Annotation Charter Standard)
    multi_answer_mode=True,               # Deactivate Stage 6 early exit; consolidate all unique answers into semicolon text
    scale_budgets_by_num_images=True,     # Linearly scale Stage 4 scan and latent budgets by N images
    
    # Scaled Budgets for N Candidate Images
    max_sam3_eval_patches=16 * NUM_CANDIDATE_IMAGES, # Scaled SAM3 scanning budget
    max_latent_evals=8 * NUM_CANDIDATE_IMAGES,       # Scaled latent speculative query budget
    max_retained_patches=32 * NUM_CANDIDATE_IMAGES,  # Scaled retention capacity before Stage 5/6
    
    # Phase 1 & 2 Early Assessment
    enable_direct_answer=True,
    direct_answer_conf_threshold=0.60,
    enable_fast_path=True,
    fast_path_conf_threshold=0.70,
    enable_visual_prompt_bbox=True,
    
    # Stage 3: Saliency-Plateau Mode
    partition_stopping_mode="saliency_plateau",
    min_plateau_size=512,
    enable_spatial_compressed_canvas=True,
    spatial_compression_gutter_px=28,
    enable_stage5_spatial_nms=True,
    stage5_nms_iou_threshold=0.60,
    max_plateau_size=1536,
    saliency_plateau_gain_threshold=0.04,
    saliency_target_threshold=0.35,
    max_patches_per_image=100,
    
    # Stage 4: Feature Grouping & Gating
    enable_feature_grouping=True,
    grouping_sim_threshold=0.80,
    latent_prompt_mode="question_aware",
    
    # Stage 6: Reading & De-duplication
    enable_early_consensus_exit=False,    # Evaluates all candidate crops across all N images
    prompt_directed_final_answer=True,
    max_vlm_tokens=128,
    answer_dedup_similarity=0.85,
    verbose=True
)

print(f"[CONFIG] MSSearch initialized in Multi-Source Mode (N={NUM_CANDIDATE_IMAGES}):")
print(f"  - max_sam3_eval_patches = {config_track_a.max_sam3_eval_patches}")
print(f"  - max_latent_evals      = {config_track_a.max_latent_evals}")
print(f"  - enable_early_consensus_exit = {config_track_a.enable_early_consensus_exit}")
print(f"  - multi_answer_mode     = {config_track_a.multi_answer_mode}")


In [ ]:
# ==============================================================================
# CELL 4: STAGE 1 — ZERO-SHOT MLLM QUESTION PARSING
# ==============================================================================
import time
import matplotlib.patches as mpatches

print("=" * 75)
print(f"STAGE 1: QUESTION PARSING (Multi-Source Track A Sample {SAMPLE_INDEX})")
print(f"Input Question: '{real_question}'")
print(f"Candidate Images Pool C: {len(candidate_images)} images (Target at Pool Index {target_pool_idx})")
print(f"[Stage 1 Diagnostics] qwen_engine: {type(qwen_engine).__name__ if qwen_engine is not None else 'None'}")
print("=" * 75)

t0 = time.perf_counter()
raw_active_objects, parser_strategy = parse_question_to_active_objects(
    question=real_question,
    zoom_model=qwen_engine,
    nlp_model=nlp_parser,
    verbose=True,
    return_strategy=True,
    image_pil=candidate_images[target_pool_idx],
    config=config_track_a
)

active_objects = list(raw_active_objects.active_objects)
question_type = getattr(raw_active_objects, "question_type", "unknown")
latency_stage1 = round(time.perf_counter() - t0, 4)

print(f"\n[Stage 1 Output] Structured Semantic Parse:")
print(f"  • Question Type           : '{question_type}'")
print(f"  • Final Active Objects A_q: {active_objects}")
print(f"[Stage 1 Telemetry] Extraction Strategy: {parser_strategy} | Latency: {latency_stage1}s\n")

fig, ax = plt.subplots(figsize=(11, 3.2))
ax.set_xlim(0, 11)
ax.set_ylim(0, 3.4)
ax.axis('off')

card = mpatches.FancyBboxPatch((0.1, 0.2), 10.8, 3.0, boxstyle="round,pad=0.2", 
                               edgecolor='#1a237e', facecolor='#f8f9fa', linewidth=2.0)
ax.add_patch(card)

ax.text(0.3, 2.55, f"[Multi-Source Sample {SAMPLE_INDEX}] {real_question}", fontsize=11, fontweight='bold', color='#1a237e')
ax.text(0.3, 1.85, f"Type: {question_type} | Strategy: {parser_strategy} | Pool Size: N={len(candidate_images)}", 
        fontsize=9.5, color='#424242', style='italic')

chips_x = 0.3
for i, obj in enumerate(active_objects):
    badge = 'Target' if i == 0 else 'Object'
    border_col = '#c2185b' if i == 0 else '#00796b'
    bg_col = '#fce4ec' if i == 0 else '#e0f2f1'
    txt_col = '#880e4f' if i == 0 else '#004d40'
    
    chip = mpatches.FancyBboxPatch((chips_x, 0.45), len(obj)*0.20 + 1.05, 0.76, 
                                   boxstyle="round,pad=0.1", edgecolor=border_col, facecolor=bg_col, linewidth=1.5)
    ax.add_patch(chip)
    ax.text(chips_x + 0.15, 0.65, f"{badge}: {obj}", fontsize=9.5, fontweight='bold', color=txt_col)
    chips_x += len(obj)*0.20 + 1.35

plt.tight_layout()
plt.show()


In [ ]:
# ==============================================================================
# CELL 5: 2-GATE EARLY ASSESSMENT ACROSS ALL N CANDIDATE IMAGES
# ==============================================================================
print("=" * 75)
print(f"2-GATE EARLY ASSESSMENT ACROSS ALL N = {len(candidate_images)} CANDIDATE IMAGES")
print(f"Active Objects A_q: {active_objects}")
print(f"Phase 1 Threshold: {config_track_a.direct_answer_conf_threshold:.2f} | Phase 2 Threshold: {config_track_a.fast_path_conf_threshold:.2f}")
print("=" * 75)

# --- GATE 1: Phase 1 Direct Global Answer Assessment ---
t0_da = time.perf_counter()
direct_ans_res = run_direct_global_assessment(
    candidate_images=candidate_images,
    question=real_question,
    zoom_model=qwen_engine,
    config=config_track_a
)
t_da = time.perf_counter() - t0_da
print(f"\n[Phase 1 Gate Result] (Latency: {t_da:.3f}s)")
print(f"  - Triggered      : {direct_ans_res.triggered}")
print(f"  - Confidence     : {direct_ans_res.confidence:.3f}")
print(f"  - Answer(s)      : {direct_ans_res.answer}")
print(f"  - Evidence Images: {getattr(direct_ans_res, 'evidence_images', [direct_ans_res.evidence_img_idx])}")

# --- GATE 2: Stage 2 Tool-Based Fast Path Assessment ---
t0_fp = time.perf_counter()
fast_path_res = run_tool_based_fast_path(
    candidate_images=candidate_images,
    active_objects=active_objects,
    question=real_question,
    sam_model=sam3_engine,
    zoom_model=qwen_engine,
    config=config_track_a
)
t_fp = time.perf_counter() - t0_fp
print(f"\n[Stage 2 Fast Path Result] (Latency: {t_fp:.3f}s)")
print(f"  - Triggered      : {fast_path_res.triggered}")
print(f"  - Confidence     : {fast_path_res.confidence:.3f}")
print(f"  - Answer(s)      : {fast_path_res.answer}")
print(f"  - Evidence Images: {getattr(fast_path_res, 'evidence_images', [fast_path_res.evidence_img_idx])}")
print(f"  - Cached Features: {list(fast_path_res.feature_cache.keys())}")


In [ ]:
# ==============================================================================
# CELL 6: STAGE 3 - MULTI-IMAGE SALIENCY-PLATEAU PARTITIONING & VISUALIZATION
# ==============================================================================
import copy
import matplotlib.patches as mpatches

print("=" * 75)
print(f"STAGE 3: MULTI-IMAGE SALIENCY-PLATEAU PARTITIONING (N = {len(candidate_images)} Images)")
print(f"Stopping Mode: '{config_track_a.partition_stopping_mode}' (gain threshold: {config_track_a.saliency_plateau_gain_threshold})")
print("=" * 75)

t0_stage3 = time.perf_counter()
feature_cache = fast_path_res.feature_cache if config_track_a.reuse_stage2_features else {}

bounded_patches = partition_candidate_images(
    candidate_images=candidate_images,
    stage2_feature_cache=feature_cache,
    config=config_track_a,
    active_objects=active_objects,
    siglip_model=siglip_engine
)
t_stage3 = time.perf_counter() - t0_stage3

print(f"\n[Stage 3 Summary]")
print(f"  - Total Bounded Patches Generated : {len(bounded_patches)}")
print(f"  - Stage 3 Partitioning Latency    : {t_stage3:.3f}s")

# Distribution per candidate image
patches_per_img = {}
for p in bounded_patches:
    patches_per_img[p.img_idx] = patches_per_img.get(p.img_idx, 0) + 1

for img_idx in range(len(candidate_images)):
    is_t = (img_idx == target_pool_idx)
    tag = "TARGET" if is_t else "DISTRACTOR"
    print(f"  - Image {img_idx} [{tag}]: {patches_per_img.get(img_idx, 0)} surviving candidate patches")

# ------------------------------------------------------------------------------
# 1. Visualize Partitioned Patches Overlaid Across All N Candidate Images
# ------------------------------------------------------------------------------
fig, axes = plt.subplots(1, len(candidate_images), figsize=(6 * len(candidate_images), 6))
if len(candidate_images) == 1:
    axes = [axes]

cols = plt.cm.tab20(np.linspace(0, 1, max(1, len(bounded_patches))))
for img_idx, ax in enumerate(axes):
    vis_img = candidate_images[img_idx].copy()
    ax.imshow(vis_img)
    img_patches = [(idx, p) for idx, p in enumerate(bounded_patches) if p.img_idx == img_idx]
    
    for idx, p in img_patches:
        x1, y1, x2, y2 = [int(round(float(c))) for c in p.box]
        c = cols[idx % len(cols)]
        rect = mpatches.Rectangle((x1, y1), x2 - x1, y2 - y1, linewidth=2.5, edgecolor=c, facecolor='none')
        ax.add_patch(rect)
        s_lbl = f"S={p.siglip_score:.2f}\n" if p.siglip_score is not None else ""
        ax.text(x1 + 10, y1 + 35, f"P{idx}\n{s_lbl}H={p.entropy:.2f}", fontsize=7.5, fontweight='bold',
                color='white', bbox=dict(boxstyle='square,pad=0.2', facecolor=c, alpha=0.85))
    
    is_t = (img_idx == target_pool_idx)
    color = "forestgreen" if is_t else "royalblue"
    ax.set_title(f"Image {img_idx} [{'TARGET' if is_t else 'DISTRACTOR'}]\n{len(img_patches)} Patches",
                 fontsize=11, fontweight="bold", color=color)
    ax.axis("off")

plt.suptitle(f"Stage 3: Partitioned Candidate Patches Across Candidate Pool C (Total: {len(bounded_patches)})", fontsize=13, fontweight="bold")
plt.tight_layout()
plt.show()

# ------------------------------------------------------------------------------
# 2. Candidate Patches Gallery (Crops Across Candidate Images)
# ------------------------------------------------------------------------------
n_show_s3 = min(len(bounded_patches), 12)
if n_show_s3 > 0:
    ncols = min(4, n_show_s3)
    nrows = (n_show_s3 + ncols - 1) // ncols
    fig, axes = plt.subplots(nrows, ncols, figsize=(4.2 * ncols, 3.8 * nrows))
    if nrows == 1 and ncols == 1:
        axes = np.array([axes])
    else:
        axes = axes.flatten()
        
    for i in range(n_show_s3):
        ax = axes[i]
        p = bounded_patches[i]
        is_t = (p.img_idx == target_pool_idx)
        crop = candidate_images[p.img_idx].crop(tuple(int(round(float(c))) for c in p.box))
        ax.imshow(crop)
        border_col = '#2e7d32' if is_t else '#1565c0'
        tag = "TARGET" if is_t else "DISTRACTOR"
        s_lbl = f" | S: {p.siglip_score:.2f}" if p.siglip_score is not None else ""
        ax.set_title(f"P{i} [Img {p.img_idx} - {tag}]\nSize: {crop.width}x{crop.height}px | H: {p.entropy:.2f}{s_lbl}",
                     fontsize=8.5, fontweight='bold', color=border_col)
        for spine in ax.spines.values():
            spine.set_edgecolor(border_col)
            spine.set_linewidth(2)
        ax.set_xticks([])
        ax.set_yticks([])
        
    for j in range(n_show_s3, len(axes)):
        axes[j].axis('off')
        
    plt.suptitle(f"Stage 3: Candidate Patches Gallery (Showing first {n_show_s3} of {len(bounded_patches)})", fontsize=12, fontweight="bold")
    plt.tight_layout()
    plt.show()


In [ ]:
# ==============================================================================
# CELL 7: STAGE 4 - SAM3 FEATURE GROUPING & SCALED ACTIVE OBJECT GATING
# ==============================================================================
import textwrap
import matplotlib.patches as mpatches

print("=" * 75)
print(f"STAGE 4: SAM3 FEATURE GROUPING & SCALED GATING (N = {len(candidate_images)} Images)")
print(f"Budgets: max_sam3_eval_patches = {config_track_a.max_sam3_eval_patches} | max_latent_evals = {config_track_a.max_latent_evals} | max_retained = {config_track_a.max_retained_patches}")
print("=" * 75)

t0_stage4 = time.perf_counter()
remaining_patches, stage4_groups = run_feature_grouping_and_gating(
    bounded_patches=bounded_patches,
    candidate_images=candidate_images,
    active_objects=active_objects,
    sam_model=sam3_engine,
    config=config_track_a,
    detected_boxes=fast_path_res.detected_boxes,
    zoom_model=qwen_engine,
    return_debug=True,
    question=real_question
)
t_stage4 = time.perf_counter() - t0_stage4

pruned_count = len(bounded_patches) - len(remaining_patches)
prune_pct = (pruned_count / max(1, len(bounded_patches))) * 100

print(f"\n[Stage 4 Summary]")
print(f"  - Total Formed Groups             : {len(stage4_groups)}")
print(f"  - Surviving Patches for Reading   : {len(remaining_patches)} / {len(bounded_patches)} ({pruned_count} pruned, {prune_pct:.1f}% reduction)")
print(f"  - Stage 4 Execution Latency       : {t_stage4:.3f}s")

# Breakdown of surviving patches per candidate image
surviving_per_img = {}
for p in remaining_patches:
    surviving_per_img[p.img_idx] = surviving_per_img.get(p.img_idx, 0) + 1

for img_idx in range(len(candidate_images)):
    is_t = (img_idx == target_pool_idx)
    tag = "TARGET" if is_t else "DISTRACTOR"
    total_s3_img = len([p for p in bounded_patches if p.img_idx == img_idx])
    print(f"  - Image {img_idx} [{tag}]: {surviving_per_img.get(img_idx, 0)} / {total_s3_img} patches retained after SAM3/VLM gating")

# ------------------------------------------------------------------------------
# 1. Visualize Original Patches Kept vs. Pruned Across All N Candidate Images
# ------------------------------------------------------------------------------
fig, axes = plt.subplots(1, len(candidate_images), figsize=(6 * len(candidate_images), 6))
if len(candidate_images) == 1:
    axes = [axes]

retained_s4_boxes = {(p.img_idx, tuple([int(round(float(c))) for c in p.box])) for p in remaining_patches}

for img_idx, ax in enumerate(axes):
    vis_img = candidate_images[img_idx].copy()
    ax.imshow(vis_img)
    img_patches = [(idx, p) for idx, p in enumerate(bounded_patches) if p.img_idx == img_idx]
    
    n_k = 0
    n_p = 0
    for idx, p in img_patches:
        box_int = [int(round(float(c))) for c in p.box]
        x1, y1, x2, y2 = box_int
        is_kept = (img_idx, tuple(box_int)) in retained_s4_boxes
        
        if is_kept:
            n_k += 1
            rect = mpatches.Rectangle((x1, y1), x2 - x1, y2 - y1, linewidth=3.5, edgecolor='#00e676', facecolor='none')
            ax.add_patch(rect)
            s_lbl = f"S={p.siglip_score:.2f}\n" if p.siglip_score is not None else ""
            ax.text(x1 + 10, y1 + 35, f"P{idx}\n{s_lbl}H={p.entropy:.2f}\nKEPT", fontsize=7.5, fontweight='bold',
                    color='black', bbox=dict(boxstyle='square,pad=0.2', facecolor='#00e676', alpha=0.9))
        else:
            n_p += 1
            rect = mpatches.Rectangle((x1, y1), x2 - x1, y2 - y1, linewidth=1.8, edgecolor='#ef5350', facecolor='none', linestyle='--', alpha=0.7)
            ax.add_patch(rect)
            ax.text(x1 + 10, y1 + 25, f"P{idx}\nPRUNED", fontsize=6.5, fontweight='bold',
                    color='white', bbox=dict(boxstyle='square,pad=0.15', facecolor='#c62828', alpha=0.6))
    
    is_t = (img_idx == target_pool_idx)
    color = "forestgreen" if is_t else "royalblue"
    label = f"Image {img_idx} [{'TARGET' if is_t else 'DISTRACTOR'}]\nKept: {n_k} | Pruned: {n_p} (from {len(img_patches)} total)"
    ax.set_title(label, fontsize=11, fontweight="bold", color=color)
    ax.axis("off")

plt.suptitle(f"Stage 4: Original Patches Kept vs. Pruned Across Candidate Pool C\n(Green: Retained {len(remaining_patches)} | Red Dashed: Pruned {pruned_count})", fontsize=13, fontweight="bold")
plt.tight_layout()
plt.show()

# ------------------------------------------------------------------------------
# 2. Detailed Semantic Group Diagnostics (Micro-Fovea, Gating & Heatmaps)
# ------------------------------------------------------------------------------
if len(stage4_groups) == 0:
    print("[Stage 4 Notice] No semantic groups formed (0 candidate patches reached Stage 4).")
else:
    print(f"\n[Stage 4 Group Diagnostics] Rendering {len(stage4_groups)} semantic groups...")
    for g in stage4_groups:
        k = g["num_patches"]
        member_indices = g.get("member_indices", list(range(k)))
        member_boxes = g.get("member_boxes", [g.get("rep_box", [0, 0, 100, 100])])
        member_entropies = g.get("member_entropies", [0.5] * k)
        sim_mat = np.array(g.get("pairwise_sim", np.ones((k, k))))
        status = g["status"]
        reason_label = g.get("match_reason", "").replace('_', ' ').title()
        det_objects = g.get("detected_objects", [])
        vlm_exist_score = g.get("vlm_exist_score", None)
        vlm_latent_score = g.get("vlm_latent_score", None)
        max_det_score = g.get("max_det_score", 0.0)
        vlm_tested_patch_idx = g.get("vlm_tested_patch_idx", None)
        vlm_prompt = g.get("vlm_prompt", "")
        
        # Determine group's source candidate image
        rep_idx = g.get("rep_idx", 0)
        source_img_idx = g.get("img_idx", bounded_patches[rep_idx].img_idx if rep_idx < len(bounded_patches) else 0)
        is_t_group = (source_img_idx == target_pool_idx)
        
        fovea_box = g.get("fovea_box", None)
        best_det_box = g.get("best_det_box", None)
        has_fovea = (fovea_box is not None)
        status_color = "#1b5e20" if status == "KEPT" else "#b71c1c"
        
        n_cols = 1 + k + (1 if has_fovea else 0) + 1
        col_ratios = [2.6] + [1.0] * k + ([1.1] if has_fovea else []) + [1.8]
        total_w = 5.4 + k * 2.3 + (2.6 if has_fovea else 0.0) + 3.0
        
        fig = plt.figure(figsize=(total_w, 5.2))
        fig.patch.set_facecolor('#ffffff')
        gs = fig.add_gridspec(1, n_cols, width_ratios=col_ratios, wspace=0.28)
        
        # 1. Left Card: Group Info, Telemetry
        ax_info = fig.add_subplot(gs[0])
        ax_info.axis('off')
        card = mpatches.FancyBboxPatch((0.02, 0.02), 0.96, 0.96, boxstyle="round,pad=0.05",
                                       edgecolor=status_color, facecolor='#fbfbfb', linewidth=2.5)
        ax_info.add_patch(card)
        
        ax_info.text(0.06, 0.92, f"GROUP {g['group_idx']}  [{status}]", fontsize=10.5, fontweight='bold', color=status_color)
        ax_info.text(0.06, 0.85, f"Patches: {k} | Source Img: {source_img_idx} [{'TARGET' if is_t_group else 'DISTRACTOR'}]",
                     fontsize=8.0, fontweight='bold', color='#1565c0')
        ax_info.text(0.06, 0.78, f"Evaluated Patch: P{vlm_tested_patch_idx if vlm_tested_patch_idx is not None else 'N/A'}",
                     fontsize=8.0, fontweight='bold', color='#333333')
        
        ax_info.text(0.06, 0.70, "[Micro-Fovea Token Boost]", fontsize=8.0, fontweight='bold', color='#6a1b9a')
        if fovea_box:
            fw = int(round(fovea_box[2] - fovea_box[0]))
            fh = int(round(fovea_box[3] - fovea_box[1]))
            ax_info.text(0.06, 0.63, f"• Foveated Window: {fw}x{fh}px", fontsize=7.4, color='#4a148c')
            ax_info.text(0.06, 0.56, "• Target Density: ~40% (vs 0.08%)", fontsize=7.4, color='#2e7d32', fontweight='bold')
        else:
            ax_info.text(0.06, 0.63, "• Full Patch View (Centroid)", fontsize=7.4, color='#616161')
            ax_info.text(0.06, 0.56, "• Token Density: Uniform", fontsize=7.4, color='#616161')
        
        ax_info.text(0.06, 0.48, "[VLM Gating Outcome]", fontsize=8.0, fontweight='bold', color='#0277bd')
        active_score = vlm_exist_score if vlm_exist_score is not None else vlm_latent_score
        score_label = "c_exist" if vlm_exist_score is not None else ("c_latent" if vlm_latent_score is not None else "SAM3 Det")
        score_str = f"{active_score:+.2f}" if active_score is not None else f"{max_det_score:.2f}"
        ax_info.text(0.06, 0.41, f"• {score_label}: {score_str}", fontsize=8.0, fontweight='bold', color='#1565c0')
        
        if vlm_prompt:
            ax_info.text(0.06, 0.33, "• Prompt Asked:", fontsize=7.4, fontweight='bold', color='#37474f')
            wrapped_prompt = textwrap.fill(f'"{vlm_prompt}"', width=34)
            ax_info.text(0.06, 0.18, wrapped_prompt, fontsize=6.8, style='italic', color='#546e7a')
        
        ax_info.text(0.06, 0.06, f"Reason: {reason_label[:32]}", fontsize=7.2, color='#616161', style='italic')
        
        # 2. Member Patches Gallery
        tested_patch_crop = None
        for p_idx in range(k):
            ax_patch = fig.add_subplot(gs[1 + p_idx])
            box = member_boxes[p_idx] if p_idx < len(member_boxes) else member_boxes[0]
            box_int = tuple(int(round(float(c))) for c in box)
            global_p_idx = member_indices[p_idx] if p_idx < len(member_indices) else p_idx
            p_img_idx = bounded_patches[global_p_idx].img_idx if global_p_idx < len(bounded_patches) else source_img_idx
            crop = candidate_images[p_img_idx].crop(box_int)
            ax_patch.imshow(crop)
            
            if global_p_idx == vlm_tested_patch_idx:
                tested_patch_crop = crop
            
            for d_obj in det_objects:
                if d_obj.get("patch_idx") == global_p_idx and "local_box" in d_obj:
                    lx1, ly1, lx2, ly2 = [int(round(float(c))) for c in d_obj["local_box"]]
                    rect = mpatches.Rectangle((lx1, ly1), lx2 - lx1, ly2 - ly1, linewidth=2.5,
                                              edgecolor='#00e5ff', facecolor='none', linestyle='-')
                    ax_patch.add_patch(rect)
                    ax_patch.text(lx1 + 8, max(18, ly1 + 18), f"[DET] {d_obj.get('label', '')[:12]}", fontsize=7.0,
                                  fontweight='bold', color='black', bbox=dict(boxstyle='square,pad=0.2', facecolor='#00e5ff', alpha=0.9))
                    
                    if fovea_box and global_p_idx == vlm_tested_patch_idx:
                        fx1, fy1, fx2, fy2 = [int(round(float(c))) for c in fovea_box]
                        rect_fov = mpatches.Rectangle((fx1, fy1), fx2 - fx1, fy2 - fy1, linewidth=2.0,
                                                      edgecolor='#ffd700', facecolor='none', linestyle='--')
                        ax_patch.add_patch(rect_fov)
            
            h_val = member_entropies[p_idx] if p_idx < len(member_entropies) else 0.5
            is_t_p = (p_img_idx == target_pool_idx)
            tag_p = "T" if is_t_p else "D"
            ax_patch.set_title(f"P{global_p_idx} [Img {p_img_idx}({tag_p})]\n(H={h_val:.2f})", fontsize=8.2, fontweight='bold')
            ax_patch.axis('off')
        
        # 3. Micro-Fovea Zoom Subplot
        curr_col = 1 + k
        if has_fovea and tested_patch_crop is not None:
            ax_fov = fig.add_subplot(gs[curr_col])
            fov_int = tuple(int(round(float(c))) for c in fovea_box)
            f_crop = tested_patch_crop.crop(fov_int)
            ax_fov.imshow(f_crop)
            ax_fov.set_title(f"MICRO-FOVEA ZOOM\n({f_crop.width}x{f_crop.height}px)", fontsize=8.2, fontweight='bold', color='#6a1b9a')
            ax_fov.axis('off')
            curr_col += 1
        
        # 4. Pairwise Cosine Similarity Heatmap
        ax_heat = fig.add_subplot(gs[curr_col])
        im = ax_heat.imshow(sim_mat, cmap='YlGnBu', vmin=0.75, vmax=1.0)
        for i in range(k):
            for j in range(k):
                val = sim_mat[i, j]
                text_col = 'white' if val > 0.90 else 'black'
                ax_heat.text(j, i, f"{val:.2f}", ha='center', va='center', fontsize=8.5, fontweight='bold', color=text_col)
        ax_heat.set_xticks(range(k))
        ax_heat.set_yticks(range(k))
        ax_heat.set_xticklabels([f"P{member_indices[i]}" if i < len(member_indices) else f"P{i}" for i in range(k)], fontsize=8)
        ax_heat.set_yticklabels([f"P{member_indices[i]}" if i < len(member_indices) else f"P{i}" for i in range(k)], fontsize=8)
        ax_heat.set_title(f"Cosine Sim\n(Mean: {sim_mat.mean():.2f})", fontsize=8.5, fontweight='bold')
        fig.colorbar(im, ax=ax_heat, fraction=0.046, pad=0.04)
        
        plt.tight_layout()
        plt.show()


In [ ]:
# ==============================================================================
# CELL 8: STAGE 5 - DIRECT UNIFIED CANDIDATE POOL AGGREGATION
# ==============================================================================
print("=" * 75)
print("STAGE 5: DIRECT UNIFIED CANDIDATE POOL AGGREGATION ACROSS N IMAGES")
print("=" * 75)

t0_stage5 = time.perf_counter()
unified_pool = run_resolution_screening(
    remaining_patches=remaining_patches,
    candidate_images=candidate_images,
    active_objects=active_objects,
    siglip_model=siglip_engine,
    config=config_track_a
)
t_stage5 = time.perf_counter() - t0_stage5

print(f"\n[Stage 5 Summary]")
print(f"  - Unified Candidate Pool P Size   : {len(unified_pool)} crops")
print(f"  - Stage 5 Screening Latency       : {t_stage5:.3f}s")

# Distribution in unified pool
pool_per_img = {}
for c in unified_pool:
    pool_per_img[c.img_idx] = pool_per_img.get(c.img_idx, 0) + 1

for img_idx in range(len(candidate_images)):
    is_t = (img_idx == target_pool_idx)
    tag = "TARGET" if is_t else "DISTRACTOR"
    print(f"  - Image {img_idx} [{tag}]: {pool_per_img.get(img_idx, 0)} candidate crops ready for Stage 6 reading")

# ------------------------------------------------------------------------------
# Visualize Retained Candidate Crops in Unified Pool P
# ------------------------------------------------------------------------------
n_show = min(len(unified_pool), 12)
if n_show > 0:
    ncols = min(4, n_show)
    nrows = (n_show + ncols - 1) // ncols
    fig, axes = plt.subplots(nrows, ncols, figsize=(4.2 * ncols, 3.8 * nrows))
    if nrows == 1 and ncols == 1:
        axes = np.array([axes])
    else:
        axes = axes.flatten()
        
    fig.suptitle(f"Stage 5: Unified Candidate Pool P ({len(unified_pool)} Total Retained Candidates)", 
                 fontsize=13, fontweight='bold', y=1.02)
    
    for i in range(n_show):
        ax = axes[i]
        cand = unified_pool[i]
        is_t = (cand.img_idx == target_pool_idx)
        crop = cand.crop_image if hasattr(cand, 'crop_image') and cand.crop_image is not None else candidate_images[cand.img_idx].crop(cand.box)
        ax.imshow(crop)
        
        score_val = getattr(cand, 'confidence_score', getattr(cand, 'score', 1.0))
        branch = getattr(cand, 'source_branch', 'Direct')
        tag = "TARGET" if is_t else "DISTRACTOR"
        border_color = '#2e7d32' if is_t else '#1565c0'
        
        ax.set_title(f"Crop {i} | Image {cand.img_idx} [{tag}]\nSize: {crop.width}x{crop.height}px | Branch: {branch}", 
                     fontsize=9, fontweight='bold', color=border_color)
        for spine in ax.spines.values():
            spine.set_edgecolor(border_color)
            spine.set_linewidth(2.5)
        ax.set_xticks([])
        ax.set_yticks([])
        
    for j in range(n_show, len(axes)):
        axes[j].axis('off')
        
    plt.tight_layout()
    plt.show()


In [ ]:
# ==============================================================================
# CELL 9: STAGE 6 - FOVEATED VLM READING & OPEN-SET MULTI-ANSWER DEDUPLICATION
# ==============================================================================
import textwrap
import matplotlib.patches as mpatches

print("=" * 75)
print(f"STAGE 6: FOVEATED VLM READING WITHOUT EARLY CONSENSUS EXIT (Sample {SAMPLE_INDEX})")
print(f"Question: '{real_question}' | Multi-Answer Mode: {config_track_a.multi_answer_mode}")
print("=" * 75)

if qwen_engine is None:
    print("\n[CRITICAL WARNING] qwen_engine (Qwen2.5-VL) is None!")
    print("  -> VLM weights are NOT loaded; Stage 6 cannot inspect image patches.")
    print("  -> Verify GPU availability and that Cell 3 ran ModelQwenVL without errors.\n")

t0_stage6 = time.perf_counter()
final_result = run_foveated_vlm_reading(
    unified_pool=unified_pool,
    candidate_images=candidate_images,
    question=real_question,
    zoom_model=qwen_engine,
    config=config_track_a,
    question_type=None
)
t_stage6 = time.perf_counter() - t0_stage6

# Prioritize informative answers over refusal / negative answers
from mssearch.stages.stage6_foveated_reader import is_negative_or_unanswerable
informative_answers = [
    a for a in final_result.answers if not is_negative_or_unanswerable(a)
]

if config_track_a.multi_answer_mode:
    consolidated_answer = "; ".join(informative_answers) if informative_answers else "; ".join(final_result.answers) if final_result.answers else "No answer found"
else:
    consolidated_answer = informative_answers[0] if informative_answers else final_result.answers[0] if final_result.answers else "No answer found"

crop_records = getattr(final_result, "crop_records", [])

print(f"\n[Stage 6 Multi-Source Reading Results]")
print(f"  - Total Crops Evaluated           : {len(crop_records)} / {len(unified_pool)}")
print(f"  - Raw Extractions Count           : {len(final_result.extractions)}")
print(f"  - Structured Unique Answers (List): {final_result.answers}")
print(f"  - Informative Filtered Answers    : {informative_answers}")
print(f"  - Consolidated Multi-Answer Text  : '{consolidated_answer}'")
print(f"  - Discovered Evidence Images      : {final_result.evidence_image_indices} (Target was at Pool Index {target_pool_idx})")
print(f"  - Stage 6 Reading Latency         : {t_stage6:.3f}s")

# Display each extraction detail
print(f"\n[Evidence Extractions Detail]")
for i, ext in enumerate(final_result.extractions):
    is_t = (ext.img_idx == target_pool_idx)
    tag = "TARGET" if is_t else "DISTRACTOR"
    print(f"  - Extraction #{i+1} [Image {ext.img_idx} - {tag}]: Final: '{ext.final_answer}' (Raw: '{ext.raw_answer[:60]}...')")

# ------------------------------------------------------------------------------
# 1. Visualize Stage 6 Evaluated Patches Overlaid Across All N Candidate Images
# ------------------------------------------------------------------------------
fig, axes = plt.subplots(1, len(candidate_images), figsize=(6 * len(candidate_images), 6))
if len(candidate_images) == 1:
    axes = [axes]

for img_idx, ax in enumerate(axes):
    vis_img = candidate_images[img_idx].copy()
    draw = ImageDraw.Draw(vis_img)
    img_records = [c for c in crop_records if c.get("img_idx") == img_idx]
    evid_count = sum(1 for c in img_records if c.get("is_evidence"))
    
    for c in img_records:
        is_ev = c.get("is_evidence", False)
        c_box = [int(round(float(v))) for v in c.get("box", [0, 0, 0, 0])]
        exp_box = [int(round(float(v))) for v in c.get("expanded_box", [0, 0, 0, 0])]
        
        if is_ev:
            draw.rectangle(c_box, outline="#00e676", width=4)
            draw.rectangle(exp_box, outline="#00e5ff", width=2)
        else:
            draw.rectangle(c_box, outline="#ff9800", width=2)
            
    thumb = vis_img.copy()
    thumb.thumbnail((768, 768))
    ax.imshow(thumb)
    is_t = (img_idx == target_pool_idx)
    color = "forestgreen" if is_t else "royalblue"
    label = f"Image {img_idx} [{'TARGET' if is_t else 'DISTRACTOR'}]\n{evid_count} Evidence / {len(img_records)} Evaluated Crops"
    ax.set_title(label, fontsize=11, fontweight="bold", color=color)
    ax.axis("off")

plt.suptitle(f"Stage 6 Evaluated Patches Overlaid on Candidate Pool C (Total Crops: {len(crop_records)} | Green: Evidence | Orange: Filtered)", 
             fontsize=13, fontweight="bold")
plt.tight_layout()
plt.show()

# ------------------------------------------------------------------------------
# 2. Visualize All Stage 6 Foveated Reading Crops & Predicted Text
# ------------------------------------------------------------------------------
n_read = len(crop_records)
if n_read > 0:
    ncols = min(4, n_read)
    nrows = (n_read + ncols - 1) // ncols
    fig, axes = plt.subplots(nrows, ncols, figsize=(4.8 * ncols, 5.0 * nrows))
    if nrows == 1 and ncols == 1:
        axes = np.array([axes])
    else:
        axes = axes.flatten()
        
    fig.suptitle(f"Stage 6: All Foveated Reading Crops & Predicted Text ({n_read} Total Crops)\nQuestion: '{real_question}'", 
                 fontsize=12.5, fontweight='bold', y=1.00)
    
    for i in range(n_read):
        ax = axes[i]
        c_rec = crop_records[i]
        img_idx = c_rec.get('img_idx', 0)
        exp_box = c_rec.get('expanded_box', [0, 0, 100, 100])
        loc_box = c_rec.get('local_target_box', None)
        is_ev = c_rec.get('is_evidence', False)
        ans = c_rec.get('raw_answer', '')
        src = c_rec.get('source_branch', '')
        is_t = (img_idx == target_pool_idx)
        
        crop_pil = candidate_images[img_idx].crop(tuple(exp_box))
        ax.imshow(crop_pil)
        
        if loc_box and len(loc_box) == 4:
            rect = mpatches.Rectangle((loc_box[0], loc_box[1]), loc_box[2] - loc_box[0], loc_box[3] - loc_box[1],
                                      linewidth=2.5, edgecolor='red', facecolor='none')
            ax.add_patch(rect)
            
        wrapped_ans = textwrap.fill(ans, width=32)
        status_tag = "EVIDENCE" if is_ev else "FILTERED"
        border_col = '#1b5e20' if is_ev else '#757575'
        tag = "TARGET" if is_t else "DISTRACTOR"
        
        ax.set_title(f"Crop {i} [Img {img_idx} - {tag}] | [{status_tag}]", fontsize=9.0,
                     color=border_col, fontweight='bold', pad=6)
        ax.set_xlabel(f'Predicted Text:\n"{wrapped_ans}"', fontsize=8.5,
                      fontweight='bold', color=border_col, labelpad=8)
        for spine in ax.spines.values():
            spine.set_edgecolor(border_col)
            spine.set_linewidth(3 if is_ev else 1.2)
        ax.set_xticks([])
        ax.set_yticks([])
        
    for j in range(n_read, len(axes)):
        axes[j].axis('off')
        
    plt.tight_layout()
    plt.show()

# ------------------------------------------------------------------------------
# 3. Stage 6 Extraction & Crop Audit Summary Table
# ------------------------------------------------------------------------------
print("=" * 110)
print(f"{'Crop #':<8} | {'Image #':<12} | {'Status':<10} | {'Branch':<10} | {'Size':<12} | {'Final Answer':<20} | {'Full Predicted VLM Response'}")
print("-" * 110)
for i, c_rec in enumerate(crop_records):
    status = "EVIDENCE" if c_rec.get("is_evidence") else "FILTERED"
    img_idx = c_rec.get("img_idx", 0)
    is_t = (img_idx == target_pool_idx)
    img_label = f"Img {img_idx} ({'T' if is_t else 'D'})"
    branch = str(c_rec.get("source_branch", "Direct"))
    box = c_rec.get("expanded_box", [0, 0, 0, 0])
    size_str = f"{box[2]-box[0]}x{box[3]-box[1]}px"
    ans = c_rec.get("raw_answer", "").strip()
    fa = str(c_rec.get('final_answer', 'N/A')).strip()
    print(f"{i:<8} | {img_label:<12} | {status:<10} | {branch:<10} | {size_str:<12} | {fa:<20} | {ans}")
print("=" * 110)

# ------------------------------------------------------------------------------
# 4. End-to-End Patch Triage Progression: Original -> Retained S4 -> Retained S6
# ------------------------------------------------------------------------------
print("\n" + "=" * 75)
print("END-TO-END PATCH TRIAGE PROGRESSION: (ORIGINAL -> RETAINED S4 -> RETAINED S6)")
print("=" * 75)

# Fast lookup for Stage 4 and Stage 6 status
s4_retained_boxes = {(p.img_idx, tuple([int(round(float(c))) for c in p.box])) for p in remaining_patches}

s6_records_map = {}
s6_evidence_boxes = set()
s6_filtered_boxes = set()

for c_rec in crop_records:
    c_img_idx = c_rec.get("img_idx", 0)
    c_box = tuple([int(round(float(v))) for v in c_rec.get("box", [0, 0, 0, 0])])
    key = (c_img_idx, c_box)
    s6_records_map[key] = c_rec
    if c_rec.get("is_evidence"):
        s6_evidence_boxes.add(key)
    else:
        s6_filtered_boxes.add(key)

# Build comprehensive lifecycle registry for all original patches
patch_lifecycle = []
for idx, p in enumerate(bounded_patches):
    p_box_int = tuple([int(round(float(c))) for c in p.box])
    key = (p.img_idx, p_box_int)
    is_kept_s4 = key in s4_retained_boxes
    
    if not is_kept_s4:
        status_s4 = "REMOVED"
        status_s6 = "N/A"
        overall = "REMOVED_STAGE_4"
        color = "#d32f2f" # Red
        c_record = None
    else:
        status_s4 = "KEPT"
        if key in s6_evidence_boxes:
            status_s6 = "EVIDENCE"
            overall = "KEPT_STAGE_6"
            color = "#2e7d32" # Emerald green
            c_record = s6_records_map.get(key)
        else:
            status_s6 = "FILTERED"
            overall = "FILTERED_STAGE_6"
            color = "#f57c00" # Amber
            c_record = s6_records_map.get(key)
            
    patch_lifecycle.append({
        "orig_idx": idx,
        "patch": p,
        "key": key,
        "is_kept_s4": is_kept_s4,
        "status_s4": status_s4,
        "status_s6": status_s6,
        "overall": overall,
        "color": color,
        "record": c_record
    })

# 4A. Canvas Progression: Side-by-Side 3 Columns per Candidate Image
n_imgs = len(candidate_images)
fig, axes = plt.subplots(n_imgs, 3, figsize=(18, 5.5 * n_imgs))
if n_imgs == 1:
    axes = np.array([axes])

for img_idx in range(n_imgs):
    is_t = (img_idx == target_pool_idx)
    img_tag = "TARGET" if is_t else "DISTRACTOR"
    img_patches = [x for x in patch_lifecycle if x['patch'].img_idx == img_idx]
    
    # Column 1: Original Partitioned Patches (Stage 3)
    ax_orig = axes[img_idx, 0]
    ax_orig.imshow(candidate_images[img_idx])
    for item in img_patches:
        p = item['patch']
        x1, y1, x2, y2 = [int(round(float(c))) for c in p.box]
        rect = mpatches.Rectangle((x1, y1), x2 - x1, y2 - y1, linewidth=2.0,
                                  edgecolor='#1976d2', facecolor='none')
        ax_orig.add_patch(rect)
        ax_orig.text(x1 + 10, y1 + 30, f"P{item['orig_idx']}", fontsize=7.5, fontweight='bold',
                     color='white', bbox=dict(boxstyle='square,pad=0.2', facecolor='#1976d2', alpha=0.85))
    ax_orig.set_title(f"Image {img_idx} [{img_tag}] — Original Patches (Stage 3)\nTotal: {len(img_patches)} Patches",
                      fontsize=10.5, fontweight='bold', color='forestgreen' if is_t else 'royalblue')
    ax_orig.axis("off")
    
    # Column 2: Stage 4 Gating (Kept in Green vs Pruned in Red Dashed)
    ax_s4 = axes[img_idx, 1]
    ax_s4.imshow(candidate_images[img_idx])
    n_kept_s4 = sum(1 for x in img_patches if x['is_kept_s4'])
    n_rem_s4 = len(img_patches) - n_kept_s4
    for item in img_patches:
        p = item['patch']
        x1, y1, x2, y2 = [int(round(float(c))) for c in p.box]
        if item['is_kept_s4']:
            rect = mpatches.Rectangle((x1, y1), x2 - x1, y2 - y1, linewidth=3.0,
                                      edgecolor='#00e676', facecolor='none')
            ax_s4.add_patch(rect)
            ax_s4.text(x1 + 10, y1 + 30, f"P{item['orig_idx']}\nKEPT", fontsize=7.0, fontweight='bold',
                       color='black', bbox=dict(boxstyle='square,pad=0.2', facecolor='#00e676', alpha=0.9))
        else:
            rect = mpatches.Rectangle((x1, y1), x2 - x1, y2 - y1, linewidth=1.5,
                                      edgecolor='#ef5350', facecolor='none', linestyle='--', alpha=0.7)
            ax_s4.add_patch(rect)
            ax_s4.text(x1 + 10, y1 + 25, f"P{item['orig_idx']}\nPRUNED", fontsize=6.0,
                       color='white', bbox=dict(boxstyle='square,pad=0.15', facecolor='#c62828', alpha=0.6))
    ax_s4.set_title(f"Stage 4: Feature Grouping & Gating\nKept: {n_kept_s4} | Removed (Pruned): {n_rem_s4}",
                    fontsize=10.5, fontweight='bold')
    ax_s4.axis("off")
    
    # Column 3: Stage 6 Reading Triage (Evidence Kept vs Filtered vs Early Pruned)
    ax_s6 = axes[img_idx, 2]
    ax_s6.imshow(candidate_images[img_idx])
    n_evid = sum(1 for x in img_patches if x['overall'] == 'KEPT_STAGE_6')
    n_filt = sum(1 for x in img_patches if x['overall'] == 'FILTERED_STAGE_6')
    for item in img_patches:
        p = item['patch']
        x1, y1, x2, y2 = [int(round(float(c))) for c in p.box]
        if item['overall'] == 'KEPT_STAGE_6':
            rect = mpatches.Rectangle((x1, y1), x2 - x1, y2 - y1, linewidth=4.0,
                                      edgecolor='#00e676', facecolor='none')
            ax_s6.add_patch(rect)
            fa = item['record'].get('final_answer', 'EVID') if item['record'] else 'EVID'
            ax_s6.text(x1 + 10, y1 + 35, f"P{item['orig_idx']} [EVIDENCE]\nAns: {fa}", fontsize=7.5, fontweight='bold',
                       color='black', bbox=dict(boxstyle='square,pad=0.25', facecolor='#00e676', alpha=0.95))
        elif item['overall'] == 'FILTERED_STAGE_6':
            rect = mpatches.Rectangle((x1, y1), x2 - x1, y2 - y1, linewidth=2.0,
                                      edgecolor='#ff9800', facecolor='none', linestyle='-')
            ax_s6.add_patch(rect)
            ax_s6.text(x1 + 10, y1 + 30, f"P{item['orig_idx']}\nFILTERED", fontsize=6.5, fontweight='bold',
                       color='white', bbox=dict(boxstyle='square,pad=0.2', facecolor='#e65100', alpha=0.85))
        else:
            rect = mpatches.Rectangle((x1, y1), x2 - x1, y2 - y1, linewidth=1.0,
                                      edgecolor='#757575', facecolor='none', linestyle=':', alpha=0.35)
            ax_s6.add_patch(rect)
            
    ax_s6.set_title(f"Stage 6: Foveated Reading Outcomes\nEvidence: {n_evid} | Filtered: {n_filt} | Early Pruned: {n_rem_s4}",
                    fontsize=10.5, fontweight='bold')
    ax_s6.axis("off")

plt.suptitle("End-to-End Patch Triage Progression: Original (Stage 3) -> Retained (Stage 4) -> Retained (Stage 6)",
             fontsize=13.0, fontweight='bold', y=0.995)
plt.tight_layout()
plt.show()

# ------------------------------------------------------------------------------
# 5. Original Patches Lifecycle Gallery (Individual Crops with Status Badges)
# ------------------------------------------------------------------------------
n_show_life = min(len(patch_lifecycle), 16)
if n_show_life > 0:
    ncols = min(4, n_show_life)
    nrows = (n_show_life + ncols - 1) // ncols
    fig, axes = plt.subplots(nrows, ncols, figsize=(4.8 * ncols, 4.4 * nrows))
    if nrows == 1 and ncols == 1:
        axes = np.array([axes])
    else:
        axes = axes.flatten()
        
    for i in range(n_show_life):
        ax = axes[i]
        item = patch_lifecycle[i]
        p = item['patch']
        is_t = (p.img_idx == target_pool_idx)
        tag = "TARGET" if is_t else "DISTRACTOR"
        box_int = tuple(int(round(float(c))) for c in p.box)
        crop = candidate_images[p.img_idx].crop(box_int)
        ax.imshow(crop)
        
        overall = item['overall']
        if overall == 'KEPT_STAGE_6':
            status_banner = "STAGE 6 EVIDENCE [KEPT]"
            border_c = '#2e7d32'
            lw = 3.5
            fa_val = item['record'].get('final_answer', 'N/A') if item['record'] else 'N/A'
            extra_msg = f'Final Ans: "{fa_val}"'
        elif overall == 'FILTERED_STAGE_6':
            status_banner = "STAGE 6 FILTERED [NO EVIDENCE]"
            border_c = '#f57c00'
            lw = 2.5
            extra_msg = "VLM Refusal / Filtered"
        else:
            status_banner = "STAGE 4 PRUNED [REMOVED]"
            border_c = '#d32f2f'
            lw = 1.8
            extra_msg = "Pruned during Grouping/Gating"
            
        s_lbl = f" | S: {p.siglip_score:.2f}" if p.siglip_score is not None else ""
        ax.set_title(f"P{item['orig_idx']} [Img {p.img_idx} - {tag}]\n{status_banner}",
                     fontsize=9.0, fontweight='bold', color=border_c, pad=6)
        ax.set_xlabel(f"Size: {crop.width}x{crop.height}px | H: {p.entropy:.2f}{s_lbl}\n{extra_msg}",
                      fontsize=8.2, fontweight='bold', color=border_c, labelpad=6)
        for spine in ax.spines.values():
            spine.set_edgecolor(border_c)
            spine.set_linewidth(lw)
            if overall == 'REMOVED_STAGE_4':
                spine.set_linestyle('--')
        ax.set_xticks([])
        ax.set_yticks([])
        
    for j in range(n_show_life, len(axes)):
        axes[j].axis('off')
        
    plt.suptitle("Original Patches Lifecycle: Status at Each Stage (Green: Evidence | Orange: Filtered | Red: Pruned S4)",
                 fontsize=12, fontweight='bold')
    plt.tight_layout()
    plt.show()

# ------------------------------------------------------------------------------
# 6. Quantitative Funnel Telemetry Breakdown Card
# ------------------------------------------------------------------------------
n_s3 = len(bounded_patches)
n_s4 = sum(1 for x in patch_lifecycle if x['is_kept_s4'])
n_s6 = sum(1 for x in patch_lifecycle if x['overall'] == 'KEPT_STAGE_6')

fig, ax = plt.subplots(figsize=(11, 3.4))
ax.axis('off')
card = mpatches.FancyBboxPatch((0.02, 0.05), 0.96, 0.90, boxstyle="round,pad=0.04",
                               edgecolor='#1a237e', facecolor='#f8f9fa', linewidth=2.0)
ax.add_patch(card)

ax.text(0.06, 0.82, "CANDIDATE PATCH TRIAGE FUNNEL (Original -> Retained Stage 4 -> Retained Stage 6)",
        fontsize=11.5, fontweight='bold', color='#1a237e')

# Step 1
ax.text(0.06, 0.60, "1. Original Patches (Stage 3):", fontsize=9.5, fontweight='bold', color='#333333')
ax.text(0.48, 0.60, f"{n_s3} Patches (100.0%)", fontsize=9.5, fontweight='bold', color='#1976d2')

# Step 2
pct_s4 = (n_s4 / max(1, n_s3)) * 100
ax.text(0.06, 0.42, "2. Retained in Stage 4 (Grouping & Gating):", fontsize=9.5, fontweight='bold', color='#333333')
ax.text(0.48, 0.42, f"{n_s4} Patches ({pct_s4:.1f}%) [Pruned: {n_s3 - n_s4} ({(100 - pct_s4):.1f}% reduction)]",
        fontsize=9.5, fontweight='bold', color='#2e7d32' if n_s4 > 0 else '#c62828')

# Step 3
pct_s6 = (n_s6 / max(1, n_s3)) * 100
ax.text(0.06, 0.24, "3. Retained in Stage 6 (Verified Evidence):", fontsize=9.5, fontweight='bold', color='#333333')
ax.text(0.48, 0.24, f"{n_s6} Patches ({pct_s6:.1f}%) [Net Compute Reduction: {(100 - pct_s6):.1f}%]",
        fontsize=9.5, fontweight='bold', color='#00e676' if n_s6 > 0 else '#e65100')

plt.tight_layout()
plt.show()


In [ ]:
# ==============================================================================
# CELL 10: END-TO-END PIPELINE EXECUTION ACROSS N CANDIDATE IMAGES
# ==============================================================================
print("=" * 75)
print(f"EXECUTING END-TO-END MULTI-SOURCE MSSearch PIPELINE (N = {len(candidate_images)} Images)")
print("=" * 75)

e2e_t0 = time.perf_counter()
e2e_response = get_mssearch_response(
    sam_model=sam3_engine,
    zoom_model=qwen_engine,
    siglip_model=siglip_engine,
    nlp_model=nlp_parser,
    annotation=target_sample,
    candidate_images=candidate_images,
    config=config_track_a
)
e2e_latency = time.perf_counter() - e2e_t0

telemetry = e2e_response.get("telemetry", {})
timings = telemetry.get("timings_sec", {})

print(f"\n[End-to-End Execution Outcome]")
print(f"  - Primary Consolidated Answer     : '{e2e_response.get('primary_answer')}'")
print(f"  - Structured Answers List         : {e2e_response.get('answers')}")
print(f"  - Route Selected                  : {telemetry.get('route')}")
print(f"  - Evidence Images                 : {e2e_response.get('evidence_images')} (Target Pool Index: {target_pool_idx})")
print(f"  - Total Evaluated Patch BBoxes    : {len(e2e_response.get('stage6_patch_boxes', []))}")
print(f"  - Total End-to-End Latency        : {e2e_latency:.2f}s")

# Stage timing breakdown table
print(f"\n[Latency Profiling Breakdown]")
headers = ["Pipeline Stage", "Device / Models", "Latency (s)", "Share (%)"]
table_data = []

stage_meta = [
    ("phase1_direct_answer", "Phase 1: Direct Global Answer", "GPU 0 (Qwen2.5-VL)"),
    ("stage1_parser", "Stage 1: Question Parsing", "GPU 0 (Qwen2.5-VL)"),
    ("stage2_fast_path", "Stage 2: Tool Fast Path & Feat Cache", "GPU 1 (SAM3) + GPU 0 (Qwen2.5-VL)"),
    ("stage3_partitioning", "Stage 3: Saliency-Plateau Partitioning", "CPU + GPU 0 (SigLIP)"),
    ("stage4_grouping_gating", "Stage 4: SAM3 Feature Grouping & Gating", "GPU 1 (SAM3) + GPU 0 (Qwen2.5-VL)"),
    ("stage5_screening", "Stage 5: Unified Candidate Pool Aggregation", "CPU Direct"),
    ("stage6_reading", "Stage 6: Foveated Reading & Multi-Answer", "GPU 0 (Qwen2.5-VL)")
]

total_recorded = sum(timings.values()) if timings else e2e_latency

for key, name, dev in stage_meta:
    lat = timings.get(key, 0.0)
    pct = (lat / total_recorded * 100.0) if total_recorded > 0 else 0.0
    table_data.append([name, dev, f"{lat:.2f}s", f"{pct:.1f}%"])

try:
    from tabulate import tabulate
    print(tabulate(table_data, headers=headers, tablefmt="github"))
except ImportError:
    for row in [headers] + table_data:
        print(f"  {row[0]:<45} | {row[1]:<30} | {row[2]:<10} | {row[3]:<10}")

print("=" * 75)
print(f"DIAGNOSTIC BENCHMARK COMPLETE FOR N = {len(candidate_images)} CANDIDATE IMAGES!")
print("=" * 75)
